# Phase 1 — EDA Exploratory Data Analysis (Analyse Exploratoire des Données) et préparation des données (FD001)

Objectif de cette première étape : charger `train_FD001.txt` et identifier les capteurs à variance nulle (non informatifs), avant de passer à la normalisation puis au lissage.

Référence colonnes (`data/readme.txt`) : `unit_number`, `time_cycles`, `op_setting_1..3`, `sensor_1..21` (26 colonnes, pas de header dans le fichier source).

### 1. Imports et noms des colonnes

On importe les librairies, puis on écrit en toutes lettres les 26 noms de colonnes du fichier NASA (le fichier brut n'a pas de ligne d'en-tête, donc il faut les fournir nous-mêmes).

In [ ]:
import pandas as pd  # pour charger et manipuler les données sous forme de tableau
import numpy as np  # fonctions numériques utilitaires (pas encore utilisé, mais import standard)
import matplotlib.pyplot as plt  # pour dessiner des graphiques (histogrammes de comparaison)

# Noms des 26 colonnes du fichier brut NASA (voir data/readme.txt) - les fichiers texte
# n'ont pas de ligne d'en-tête, donc on doit fournir les noms de colonnes nous-mêmes
COLUMNS = [
    "unit_number",   # identifiant du moteur : 1 à 100 pour FD001 train
    "time_cycles",   # numéro du cycle pour ce moteur, commence à 1
    "op_setting_1",  # réglage opérationnel 1
    "op_setting_2",  # réglage opérationnel 2
    "op_setting_3",  # réglage opérationnel 3
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21",  # 21 mesures de capteurs au total
]

### 2. Fonction de chargement

On définit `DATA_DIR` (le dossier où sont rangées les données, en chemin relatif depuis `notebooks/`) et une fonction `load_fd001` qui construit le bon chemin de fichier et lit le texte brut avec pandas. Une seule fonction pour charger `train` ou `test`, selon la valeur qu'on lui passe.

In [ ]:
DATA_DIR = "../data"  # chemin relatif depuis notebooks/ vers le dossier data


def load_fd001(split):
    """
    Charge le fichier train ou test de FD001 dans un DataFrame pandas.

    Parameters
    ----------
    split : str
        "train" ou "test" - sélectionne quel fichier brut charger.

    Returns
    -------
    pandas.DataFrame
        Une ligne par (moteur, cycle), avec les 26 colonnes nommées.
    """
    path = f"{DATA_DIR}/{split}_FD001.txt"  # construit le chemin complet du fichier
    df = pd.read_csv(
        path,
        sep=r"\s+",     # le fichier brut est séparé par des espaces (largeur variable), pas des virgules
        header=None,    # le fichier brut n'a pas de ligne d'en-tête
        names=COLUMNS,  # donc on fournit les noms de colonnes nous-mêmes
    )
    return df  # renvoie le tableau chargé

### 3. Chargement du fichier d'entraînement

On appelle `load_fd001` avec `"train"`, et on vérifie la taille du tableau obtenu (doit être 20 631 lignes, 26 colonnes).

In [ ]:
train = load_fd001("train")  # charge le fichier d'entraînement (100 moteurs, jusqu'à la panne)
train.shape  # vérifie la taille : doit être (20631, 26)

### 4. Contrôle visuel

On affiche les 5 premières lignes pour vérifier que les colonnes et les valeurs ont l'air correctes avant d'aller plus loin.

In [ ]:
train.head()#renvoie les 5 premières lignes du tableau train (5 par défaut). head Donne juste echnatillon pour que ca te paraisse correct

## Capteurs à variance nulle

Un capteur constant sur l'ensemble des moteurs ne peut pas être corrélé à la dégradation (il ne varie jamais) : on le retire.

**Attention à la précision flottante** : un test naïf `variance == 0` peut louper un capteur réellement constant si le calcul de variance introduit une erreur d'arrondi (observé ici sur `sensor_5` et `sensor_16` : valeur unique, mais variance calculée ≈ 1e-29 / 1e-35 au lieu de 0 exact). Le critère fiable est le nombre de valeurs distinctes (`nunique() == 1`), pas la variance.

### 5. Statistiques par capteur

Pour chacun des 21 capteurs, on calcule sa variance et son nombre de valeurs distinctes, et on met les deux côte à côte dans un tableau trié.

In [ ]:
# Construit la liste des noms de colonnes capteurs (sensor_1 à sensor_21) à partir de COLUMNS,
# avec une boucle simple plutôt qu'une comprehension compacte - plus facile à lire ligne par ligne
sensor_cols = []  # on part d'une liste vide
for column_name in COLUMNS:  # on parcourt chaque nom de colonne, dans l'ordre
    if column_name.startswith("sensor_"):  # on ne garde que les colonnes capteurs
        sensor_cols.append(column_name)  # on l'ajoute à notre liste

variance_per_sensor = train[sensor_cols].var()  # variance de chaque capteur, sur tous moteurs/cycles
unique_values_per_sensor = train[sensor_cols].nunique()  # nombre de valeurs distinctes par capteur

# Met les deux statistiques côte à côte dans un seul tableau, pour pouvoir les comparer directement
sensor_summary = pd.DataFrame({
    "variance": variance_per_sensor,
    "n_unique": unique_values_per_sensor,
})
sensor_summary = sensor_summary.sort_values("n_unique")  # les capteurs à 1 seule valeur apparaissent en premier

sensor_summary  # affiche le tableau

### 6. Liste finale des capteurs constants

On parcourt le tableau ci-dessus et on garde uniquement les capteurs dont `n_unique == 1`.

In [ ]:
# Un capteur est réellement constant s'il ne prend qu'UNE seule valeur distincte (n_unique == 1).
# On utilise n_unique ici, pas variance == 0, à cause du problème d'arrondi flottant expliqué
# ci-dessus (sensor_5 et sensor_16 sont constants mais leur variance n'est pas exactement 0)
constant_sensors = []  # on part d'une liste vide
for sensor_name in sensor_summary.index:  # on parcourt chaque capteur
    n_unique = sensor_summary.loc[sensor_name, "n_unique"]  # on récupère son nombre de valeurs distinctes
    if n_unique == 1:  # s'il ne prend jamais plus d'une valeur
        constant_sensors.append(sensor_name)  # il est constant : on l'ajoute à la liste

constant_sensors  # affiche la liste finale des capteurs constants à retirer

### 7. Suppression des capteurs constants

On retire maintenant les 6 capteurs identifiés ci-dessus du tableau `train`, puisqu'ils ne peuvent apporter aucune information utile au modèle.

In [ ]:
# .drop(columns=...) renvoie un NOUVEAU tableau sans ces colonnes - il ne modifie pas
# train sur place, donc on réaffecte le résultat à train pour que le changement compte
train = train.drop(columns=constant_sensors)

train.shape  # vérifie la nouvelle taille : 26 - 6 = 20 colonnes restantes

### 8. Vérification

On recompte les colonnes capteurs restantes dans le tableau mis à jour : ça doit tomber sur 15 (21 capteurs de départ moins les 6 retirés).

In [ ]:
# On refait le même comptage que dans la cellule 5, mais sur le tableau train mis à jour
sensor_cols_restants = []  # on part d'une liste vide
for column_name in train.columns:  # on parcourt les colonnes du tableau actuel
    if column_name.startswith("sensor_"):  # on ne garde que les colonnes capteurs
        sensor_cols_restants.append(column_name)  # on l'ajoute à la liste

len(sensor_cols_restants)  # doit valoir 15 (21 capteurs de départ - 6 retirés)

## Comparaison de deux méthodes de normalisation

Plutôt que de choisir une méthode par convention seule, on compare deux méthodes sur nos vraies données avant de trancher :

- **Min-max** : `(valeur - min) / (max - min)` — ramène chaque capteur entre 0 et 1. Très sensible aux valeurs extrêmes : une seule valeur très éloignée du reste écrase toute l'échelle et compresse le reste des valeurs dans un petit coin de [0, 1].
- **Standardisation (`StandardScaler`)** : `(valeur - moyenne) / écart-type` — centre chaque capteur sur 0 avec un écart-type de 1. Moins sensible aux valeurs extrêmes.

Critère de comparaison possible **sans modèle entraîné** : regarder si nos capteurs ont des valeurs extrêmes qui justifieraient de préférer la standardisation, via des histogrammes avant/après. Le choix final sera revalidé en Phase 4 avec un vrai chiffre (RMSE / score NASA), mais on peut déjà éliminer une méthode manifestement inadaptée à nos données.

### 9. Deux copies indépendantes de `train`

`.copy()` crée un tableau totalement indépendant en mémoire. Sans ça, écrire `train_minmax = train` ferait pointer les deux noms vers **le même** tableau — modifier l'un modifierait l'autre aussi. On veut comparer les deux méthodes séparément, donc on part de deux copies distinctes.

In [ ]:
train_minmax = train.copy()  # copie indépendante, dédiée à la méthode min-max
train_standard = train.copy()  # copie indépendante, dédiée à la méthode standardisation

### 10. Application du min-max sur `train_minmax`

Pour chacun des 15 capteurs, on calcule son min et son max sur `train_minmax`, puis on réécrit ses valeurs à l'échelle [0, 1].

In [ ]:
for sensor_name in sensor_cols_restants:  # on parcourt les 15 capteurs restants
    minimum = train_minmax[sensor_name].min()  # valeur minimale de ce capteur
    maximum = train_minmax[sensor_name].max()  # valeur maximale de ce capteur
    train_minmax[sensor_name] = (train_minmax[sensor_name] - minimum) / (maximum - minimum)  # remet à l'échelle [0, 1]

train_minmax[sensor_cols_restants].describe().loc[["min", "max"]]  # verification rapide : min=0, max=1 partout

### 11. Application de la standardisation sur `train_standard`

Pour chacun des 15 capteurs, on calcule sa moyenne et son écart-type sur `train_standard`, puis on centre-réduit ses valeurs.

In [ ]:
for sensor_name in sensor_cols_restants:  # on parcourt les 15 capteurs restants
    moyenne = train_standard[sensor_name].mean()  # moyenne de ce capteur
    ecart_type = train_standard[sensor_name].std()  # écart-type de ce capteur
    train_standard[sensor_name] = (train_standard[sensor_name] - moyenne) / ecart_type  # centre-réduit

train_standard[sensor_cols_restants].describe().loc[["mean", "std", "min", "max"]]  # verification : mean≈0, std≈1

### 12. Comparaison visuelle

On trace, pour deux capteurs (`sensor_9`, la plus grande variance brute, et `sensor_11`, une variance plus modeste), l'histogramme des valeurs brutes puis des deux versions normalisées — pour voir si l'une des deux méthodes écrase visiblement les données à cause d'une valeur extrême.

In [ ]:
capteurs_a_comparer = ["sensor_9", "sensor_11"]  # les deux capteurs qu'on affiche

fig, axes = plt.subplots(2, 3, figsize=(12, 6))  # grille 2 lignes (1 par capteur) x 3 colonnes (1 par version)

for i, sensor_name in enumerate(capteurs_a_comparer):  # on parcourt les 2 capteurs, i = 0 puis 1
    axes[i, 0].hist(train[sensor_name], bins=30)  # histogramme des valeurs brutes
    axes[i, 0].set_title(f"{sensor_name} - brut")

    axes[i, 1].hist(train_minmax[sensor_name], bins=30)  # histogramme après min-max
    axes[i, 1].set_title(f"{sensor_name} - min-max")

    axes[i, 2].hist(train_standard[sensor_name], bins=30)  # histogramme après standardisation
    axes[i, 2].set_title(f"{sensor_name} - standardisé")

plt.tight_layout()  # évite que les titres se chevauchent
plt.show()  # affiche la figure

### 13. Décision finale : on retient le min-max

Justification complète discutée hors notebook (voir `docs/01_eda_preparation.md` une fois la Phase 1 validée). En résumé : coût nul pour les modèles à arbres de la Phase 3, entrées bornées adaptées au LSTM prévu ensuite, hypothèse de distribution gaussienne de la standardisation de toute façon mise à mal par nos données (`sensor_9` très asymétrique), convention déjà établie sur ce dataset. `train_standard` est gardé au-dessus seulement comme référence de comparaison, pas réutilisé pour la suite.

On garde aussi une copie `train_avant_lissage`, prise ici avant le lissage — **important** : sans un nouveau `.copy()`, cette copie pointerait vers le même tableau que `train` et se retrouverait modifiée par le lissage elle aussi (même piège qu'à la cellule 9), ce qui empêcherait toute comparaison avant/après plus bas.

In [ ]:
train = train_minmax  # on continue avec la version min-max pour la suite du notebook
train_avant_lissage = train.copy()  # copie a part, gelee ici, uniquement pour comparer avant/apres lissage

## Lissage du signal

Les capteurs contiennent du bruit de mesure cycle-à-cycle, en plus du vrai signal de dégradation (ex. `sensor_2` : 641.82 → 642.15 → 642.35 → 642.35 → 642.37 sur les 5 premiers cycles du moteur 1 — cette gigue n'est pas de la dégradation). Le lissage atténue ce bruit pour que le modèle voie mieux la tendance réelle.

**Deux règles de correction à respecter, quelle que soit la méthode** :
1. **Causalité** : le lissage du cycle `t` ne doit utiliser que des cycles `≤ t` (passé), jamais de cycles futurs — sur le jeu de test, on n'a par définition pas accès au futur d'un moteur en cours de vie.
2. **Par moteur, jamais à travers deux moteurs différents** : `train` empile les 100 moteurs les uns après les autres ; un lissage naïf sur toute la colonne mélangerait la fin d'un moteur avec le début du suivant.

**Méthode retenue** : moyenne mobile, fenêtre de 5 cycles (cohérent avec le benchmark exploratoire documenté dans le README). Simple à justifier, et `pandas.rolling()` est causal par défaut (ne regarde que le passé).

### 14. Application de la moyenne mobile, moteur par moteur

`train["unit_number"].unique()` renvoie la liste des identifiants de moteurs distincts (1 à 100). Pour chaque moteur, `train["unit_number"] == moteur_id` crée une colonne de Vrai/Faux (Vrai sur les lignes de ce moteur) ; `train.loc[masque, sensor_name]` sélectionne uniquement ces lignes. `.rolling(5, min_periods=1).mean()` moyenne chaque valeur avec ses 4 précédentes (seulement dans cette sélection, donc jamais avec un autre moteur) ; `min_periods=1` évite des cases vides sur les tout premiers cycles de chaque moteur, où 5 cycles ne sont pas encore disponibles.

In [ ]:
FENETRE_LISSAGE = 5  # nombre de cycles utilisés pour la moyenne mobile

for sensor_name in sensor_cols_restants:  # on lisse chaque capteur, un par un
    for moteur_id in train["unit_number"].unique():  # on parcourt chaque moteur séparément
        masque = train["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur, Faux sinon
        # rolling() ne regarde que les lignes de cette sélection (un seul moteur) : jamais de fuite
        # entre deux moteurs, et jamais de cycle futur (rolling est causal par défaut)
        train.loc[masque, sensor_name] = train.loc[masque, sensor_name].rolling(FENETRE_LISSAGE, min_periods=1).mean()

train[sensor_cols_restants].head()  # aperçu du résultat

### 15. Vérification visuelle

On trace `sensor_11`, brut vs lissé, sur les cycles du moteur 1 seul (en utilisant `train_avant_lissage` pour le "avant", justement pour éviter le piège de la cellule 13) — le lissé doit suivre la même tendance que le brut mais avec moins de gigue cycle-à-cycle.

In [ ]:
masque_moteur_1 = train["unit_number"] == 1  # Vrai uniquement sur les lignes du moteur 1
cycles_moteur_1 = train.loc[masque_moteur_1, "time_cycles"]  # numéros de cycle du moteur 1

plt.figure(figsize=(10, 4))
plt.plot(cycles_moteur_1, train_avant_lissage.loc[masque_moteur_1, "sensor_11"], label="avant lissage", alpha=0.5)
plt.plot(cycles_moteur_1, train.loc[masque_moteur_1, "sensor_11"], label="après lissage")
plt.xlabel("cycle")
plt.ylabel("sensor_11 (normalisé)")
plt.title("Moteur 1 - sensor_11 - avant / après lissage")
plt.legend()
plt.show()